# Encoder-Decoder Models

A study refresher. An **encoder-decoder** (a.k.a. **seq2seq**) splits the model in two: an **encoder** reads the whole input and compresses it into a representation, and a **decoder** generates the output one step at a time, conditioned on that representation. It's the architecture behind translation, summarization, and speech-to-text.

**Domain:** Architectures  ·  **runnable:** yes

## 1. What & Why

**What it is.** Two stacked networks with a clean division of labor:

- **Encoder** — consumes the entire source sequence `x = (x₁…xₙ)` *at once* and turns it into a **memory**: either a single context vector (classic RNN seq2seq) or, in the Transformer, a sequence of `n` contextual vectors. It is **non-autoregressive** and bidirectional — it may look both left and right.
- **Decoder** — generates the target `y = (y₁…yₘ)` **autoregressively**, one token at a time, where each step is conditioned on (a) the tokens it has produced so far and (b) the encoder's memory via **cross-attention**.

**The problem it solves.** Many tasks map one sequence to *another* sequence of **different length and structure**: English→French, article→summary, audio→transcript, question→answer. A plain classifier or a single autoregressive LM doesn't cleanly separate "understand the input" from "produce the output." The encoder-decoder split lets the input be processed in full (bidirectionally, in parallel) while the output is produced left-to-right.

**When to reach for it.** Conditional generation where a *distinct* source must be fully understood before generating a *transformed* target: machine translation, summarization, grammatical error correction, data-to-text, speech recognition (Whisper), TTS.

**When NOT to.** If input and output live in the *same* stream and you just continue it (chat, code completion, open-ended generation), a **decoder-only** LM (GPT-style) is simpler and scales better — you prepend the input as a prompt and let one stack do everything. If you only need to *understand* and not generate (classification, retrieval, NER), an **encoder-only** model (BERT) is enough. See §7.

## 2. Mental Model

**A translator with a notepad.**

1. **Read & take notes (encode).** A human translator reads the *entire* source sentence first — looking forward and backward to resolve ambiguity ("bank" = river or money?) — and jots structured notes. That's the encoder producing **memory**; it sees the whole input, so it can be bidirectional and run in parallel.
2. **Write while glancing back (decode).** Then they write the translation **word by word**, and for each word they glance back at the relevant part of their notes. That glance is **cross-attention**: the decoder's query looks into the encoder's keys/values. They can't see words they haven't written yet — that's the decoder's **causal self-attention**.

So a Transformer decoder layer has **three** sublayers (vs the encoder's two):

```
ENCODER layer                 DECODER layer
  self-attention (bidir)        masked self-attention (causal)
  feed-forward                  cross-attention  -> into encoder memory
                                feed-forward
```

The information bottleneck of the *classic* RNN seq2seq — cramming the whole source into one fixed vector — is exactly what **attention** fixed: instead of one context vector, the decoder gets dynamic access to *all* encoder states.

## 3. Key Concepts

- **Seq2seq.** The task shape: variable-length input → variable-length output. Encoder-decoder is the architecture that implements it.
- **Encoder memory / context.** What the encoder hands the decoder. RNN seq2seq: a single final hidden state (the bottleneck). Transformer: a full `(n, d)` matrix of per-token vectors — no bottleneck.
- **Cross-attention (a.k.a. encoder-decoder attention).** The bridge. Decoder positions form **queries**; encoder outputs form **keys and values**. This is the *only* place the source influences generation in a Transformer enc-dec.
- **Causal (masked) self-attention** in the decoder. A position may attend only to itself and earlier positions, so training with all targets visible still matches autoregressive inference.
- **Teacher forcing.** During training the decoder is fed the *ground-truth* previous tokens (not its own predictions), so the whole target can be scored in **one parallel forward pass**. Cheap and stable, but creates **exposure bias** — at inference the model must consume its *own* (possibly wrong) tokens.
- **Autoregressive decoding.** At inference you generate token by token, feeding each prediction back in, until an **EOS** token or a length cap. Greedy / beam search / sampling are the strategies.
- **BOS / EOS / special tokens.** Decoding starts from a start symbol (or `decoder_start_token_id`) and stops at EOS. Getting these wrong is a classic bug.
- **The lineage.** RNN seq2seq (Sutskever 2014) → + attention (Bahdanau 2015) → Transformer enc-dec (Vaswani 2017) → pretrained enc-dec LMs: **T5, BART, mT5, Whisper**.

## 4. Setup

The two from-scratch examples need only **NumPy**, so they run anywhere on CPU — no GPU, no downloads, fully deterministic. The final example uses Hugging Face **`transformers`** with a real pretrained encoder-decoder (T5); it downloads weights, so it is **gated behind `os.getenv("RUN_HF")`** and prints the call shape otherwise — the notebook still executes top-to-bottom either way.

```bash
pip install numpy
pip install "transformers>=4.40" torch   # optional — only for the gated Example 3
```

In [1]:
import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)
print("NumPy", np.__version__, "ready — encoder-decoder demos run on CPU.")

NumPy 2.5.0 ready — encoder-decoder demos run on CPU.


## 5. Worked Examples

### Example 1 — A Transformer encoder-decoder forward pass (cross-attention)

We wire up the *mechanics* with crafted weights so the data flow is visible: the encoder turns a 4-token source into a memory matrix, then a decoder step issues a **query** that cross-attends into that memory. We aim the query at one source position so you can watch cross-attention concentrate there — that "glance back at the notes" from the mental model.

In [2]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)          # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    """Scaled dot-product attention. Q:(nq,d) K,V:(nk,d). Returns (out, weights)."""
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)                  # (nq, nk)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    w = softmax(scores, axis=-1)
    return w @ V, w

d = 8                                                # model dim

# --- ENCODER: read a 4-token source into memory (bidirectional self-attention) ---
src = rng.standard_normal((4, d))                    # source token embeddings
# One bidirectional self-attention block (no mask) gives contextual memory.
enc_memory, _ = attention(src, src, src)             # (4, d) -> encoder K/V for the decoder
print("encoder memory shape:", enc_memory.shape, "(4 source positions x d)")

# --- DECODER step: a query cross-attends into the encoder memory ---
# Aim this decoder query straight at source position 1's memory vector.
dec_query = enc_memory[1] * 3.0
ctx, cross_w = attention(dec_query[None, :], enc_memory, enc_memory)

print("cross-attention weights over the 4 source tokens:", cross_w[0])
print("argmax source position:", cross_w[0].argmax(), "(we aimed at position 1)")

encoder memory shape: (4, 8) (4 source positions x d)
cross-attention weights over the 4 source tokens: [0.    0.989 0.002 0.009]
argmax source position: 1 (we aimed at position 1)


### Example 2 — Autoregressive decoding: teacher forcing vs. free-running

Training and inference feed the decoder *different* inputs, and this is the source of endless bugs. We model a tiny deterministic "translator" whose job is to **reverse** the source sequence, and show the two regimes side by side:

- **Teacher forcing** (training): feed the *gold* previous tokens, score the whole target in one shot.
- **Autoregressive** (inference): feed the model's *own* previous output, generate until **EOS**.

The "model" here is a lookup table standing in for a trained network, so the control flow — not the weights — is the point.

In [3]:
BOS, EOS = "<bos>", "<eos>"

# A toy "trained" next-token function: given the source and what's decoded so far,
# emit the next reversed-source token, then EOS. (Stands in for a real decoder.)
def next_token(source, decoded_so_far):
    target = list(reversed(source)) + [EOS]          # the function it has "learned"
    return target[len(decoded_so_far)]               # position to emit next

source = ["the", "cat", "sat"]
gold   = list(reversed(source)) + [EOS]              # ["sat","cat","the","<eos>"]

# --- Teacher forcing: feed GOLD prefixes; one parallel pass scores every step ---
tf_inputs, tf_preds = [], []
prefix = [BOS]
for t in range(len(gold)):
    tf_inputs.append(list(prefix))
    tf_preds.append(next_token(source, prefix[1:]))  # prediction at step t
    prefix.append(gold[t])                           # next input = GOLD token
print("teacher forcing — decoder is fed gold prefixes:")
for inp, pred in zip(tf_inputs, tf_preds):
    print(f"  in={inp!s:<34} -> predict {pred}")

# --- Autoregressive: feed the model's OWN tokens until it emits EOS ---
generated, prefix = [], [BOS]
for _ in range(10):                                  # length cap = safety net
    tok = next_token(source, generated)
    if tok == EOS:
        break
    generated.append(tok)
    prefix.append(tok)
print("\nautoregressive — decoder consumes its own output:")
print("  generated:", generated, "| matches gold?", generated == gold[:-1])

teacher forcing — decoder is fed gold prefixes:
  in=['<bos>']                          -> predict sat
  in=['<bos>', 'sat']                   -> predict cat
  in=['<bos>', 'sat', 'cat']            -> predict the
  in=['<bos>', 'sat', 'cat', 'the']     -> predict <eos>

autoregressive — decoder consumes its own output:
  generated: ['sat', 'cat', 'the'] | matches gold? True


### Example 3 (optional) — A real pretrained encoder-decoder (T5), gated

`T5` is a production encoder-decoder. One `model.generate(...)` call runs the encoder over the input, then autoregressively decodes with cross-attention and beam/greedy search — all the machinery from Examples 1–2, trained. This cell downloads weights, so it only runs when `RUN_HF=1`; otherwise it prints the exact call shape and moves on, keeping the notebook executable offline.

In [4]:
import os

if os.getenv("RUN_HF"):
    from transformers import T5Tokenizer, T5ForConditionalGeneration

    tok = T5Tokenizer.from_pretrained("t5-small")
    model = T5ForConditionalGeneration.from_pretrained("t5-small")  # ~240MB

    prompt = "translate English to German: The house is wonderful."
    ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(ids, max_new_tokens=20, num_beams=4)       # encode -> decode
    print("T5:", tok.decode(out[0], skip_special_tokens=True))
else:
    print("Set RUN_HF=1 to run the real T5 encoder-decoder. Call shape:")
    print('  tok   = T5Tokenizer.from_pretrained("t5-small")')
    print('  model = T5ForConditionalGeneration.from_pretrained("t5-small")')
    print('  ids   = tok("translate English to German: ...", return_tensors="pt").input_ids')
    print("  out   = model.generate(ids, num_beams=4)   # encoder runs once, decoder loops")
    print("  -> expected output: 'Das Haus ist wunderbar.'")

Set RUN_HF=1 to run the real T5 encoder-decoder. Call shape:
  tok   = T5Tokenizer.from_pretrained("t5-small")
  model = T5ForConditionalGeneration.from_pretrained("t5-small")
  ids   = tok("translate English to German: ...", return_tensors="pt").input_ids
  out   = model.generate(ids, num_beams=4)   # encoder runs once, decoder loops
  -> expected output: 'Das Haus ist wunderbar.'


## 6. Gotchas & Pitfalls

- **`decoder_start_token_id` / BOS mistakes.** The decoder must be primed with the right start symbol (T5 uses `pad` as the start token; BART uses EOS-then-BOS). Get it wrong and generation is garbage from token one. When fine-tuning, libraries auto-build decoder inputs by *shifting* labels right — don't shift them yourself too.
- **Forgetting the causal mask in the decoder.** Without it, the decoder's self-attention sees future target tokens during teacher forcing and learns nothing useful — train loss looks great, inference collapses. (Cross-attention into the encoder is *unmasked* — that's correct; the source is fully visible.)
- **Exposure bias.** Teacher forcing trains on gold prefixes but inference consumes the model's own (sometimes wrong) tokens, so errors compound. It's inherent; mitigations include scheduled sampling and sequence-level objectives.
- **No EOS / no length cap.** If the model never emits EOS, greedy/beam decoding runs to `max_length` and produces rambling or repetitive text. Always set `max_new_tokens` *and* train EOS properly.
- **Repetition & degeneration.** Greedy and beam search love to loop ("the the the"). Reach for `no_repeat_ngram_size`, repetition penalties, or sampling (top-k/top-p) for open-ended outputs.
- **Cross-attention is the only source link (Transformer).** If you accidentally drop or zero the encoder memory, the decoder silently becomes an unconditional language model — fluent output that ignores the input. Sanity-check that outputs actually depend on the source.
- **Padding leaks via attention.** Both the encoder self-attention and the decoder's cross-attention need the source **padding mask**; without it, pad tokens contaminate the memory and the glances back.
- **Classic RNN seq2seq bottleneck.** Without attention, a single fixed context vector throttles long inputs. If you're reimplementing seq2seq from a pre-2015 tutorial, add attention — it's the whole point.
- **Tokenizer/decoder mismatch.** Decoding with a different tokenizer/vocab than the model was trained on yields valid-looking but wrong text. Keep tokenizer and model paired.

## 7. When to Use vs Alternatives

| Architecture | Shape | Strength | Weakness | Reach for it when |
|---|---|---|---|---|
| **Encoder-decoder** (T5, BART, Whisper) | seq → seq | Bidirectional input understanding + clean conditional generation; cross-attention separates "read" from "write" | Two stacks to train/serve; more params for the same width | Source and target are **distinct** sequences: translation, summarization, ASR, data-to-text |
| **Decoder-only** (GPT, Llama) | prompt → continuation | One stack; scales beautifully; in-context learning; simplest serving | Input is processed left-to-right (no native bidirectional encoding); long prompts re-encoded each call | Chat, completion, open-ended generation; "prompt + answer" fits one stream |
| **Encoder-only** (BERT, RoBERTa) | seq → labels/vectors | Deep bidirectional understanding; cheap | Can't generate text | Classification, NER, embeddings, retrieval, extractive QA |
| **Classic RNN seq2seq** (no attention) | seq → seq | Tiny, simple, streaming | Fixed-vector bottleneck; sequential; weak on long inputs | Teaching/baselines, very short sequences, tight memory |

**The big shift:** modern LLMs are mostly **decoder-only** because one stack + scale + prompting handles most tasks, including translation, via in-context learning. Encoder-decoder still wins where you genuinely need a strong **bidirectional read of a fixed source** before generating — Whisper (audio→text) and T5-style fine-tuned summarizers/translators are the durable examples.

Cross-link: [`transformer`](transformer.ipynb), [`attention-mechanisms`](attention-mechanisms.ipynb), [`t5`](t5.ipynb), [`bert`](bert.ipynb), [`rnn`](rnn.ipynb).

## 8. Resources

- **"Sequence to Sequence Learning with Neural Networks"** (Sutskever et al., 2014) — the original RNN encoder-decoder for translation: https://arxiv.org/abs/1409.3215
- **"Neural Machine Translation by Jointly Learning to Align and Translate"** (Bahdanau et al., 2015) — added attention and killed the fixed-vector bottleneck: https://arxiv.org/abs/1409.0473
- **"Attention Is All You Need"** (Vaswani et al., 2017) — the Transformer encoder-decoder, with cross-attention in the decoder: https://arxiv.org/abs/1706.03762
- **"Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer"** (T5, Raffel et al., 2020) — every NLP task as seq2seq: https://arxiv.org/abs/1910.10683
- **The Illustrated Transformer** (Jay Alammar) — visual walkthrough of the encoder-decoder stack and cross-attention: https://jalammar.github.io/illustrated-transformer/
- **Hugging Face — encoder-decoder models** — practical API for T5/BART/Whisper generation: https://huggingface.co/docs/transformers/model_doc/encoder-decoder